# Datamine PTCLD2WF Process Example

This notebook demonstrates how to configure and run the **`ptcld2wf`** process wrapper in `dmstudio`.

## Process Description

## PTCLD2WF

**PTCLD2WF** reconstructs surface data files from input point cloud files.

A single input points file containing surveyed point data is expected. Point density and arrangement can be uniform or irregular. Points can optionally be declustered and, if normal calculation is required, these stages can be performed independently of surfacing or as one combined operation, depending on _@RUNMODE_.

The **[Point Reconstruction Console](<../COMMON/point-reconstruction-console.md>)** provides a workflow-based approach to setting up parameters for **PTCLD2WF**. It also offers automated file loading and scenario management and access to an additional reconstruction method (you can also record **PTCLD2WF** macros if one of the supported methods is selected). Unless you require a process for your point reconstruction project, it is a recommended alternative.

**PTCLD2WF** processing involves up to 3 phases, depending on the wireframe surfacing method chosen:

1. Input points are declustered to decrease point density and encourage a more uniform point layout.

2. Input point data is appended with normal information, indicating the normal direction to be considered when specific surfacing methods are used. Not all methods require point normal calculation.

Normal data can also be extracted from the input points file if it is present.

3. A surface is interpolated between points using a combination of surface method and appropriate parameters.

In all cases, the input file is a Datamine points file and the output files are a wireframe file pair.

* **Note** (**PTCLD2WF** parameters can be dependent on other settings. For example, @DEPTH, @WIDTH, @SAMPNODE, @PTWEIGHT and @BOUNDTYP are only used if @WFMETHOD is 1 or 2. Similarly, if @WIDTH is greater than zero, @DEPTH is not used.):

Another example: if @WFMETHOD is not 1, @BOUNDTYP is not used.

### Input Files:

* **in** (Points):
  Input point cloud data file used to create the wireframe surface.
  Required=Yes

### Output Files:

* **points** (Points):
  Output file prefix for subsampled points and points with normals. A file containing
  subsampled points will have an _SS suffix. Points with normals will have a _SSNORM
  suffix. This must be specified if _@RUNMODE_ = 1,2 or 4.
  Required=No

* **wiretr** (Wireframe Points):
  Output surface wireframe triangle file.
  Required=Yes

* **wirept** (Wireframe Triangles):
  Output surface wireframe point file.
  Required=Yes

### Fields:

* **red** (Any : IN):
  Name of red colour field in input file IN. Only used if @WFMETHOD is 1 or 2.
  Default=Undefined
  Required=No

* **green** (Any : IN):
  Name of green colour field in input file IN. Only used if @WFMETHOD is 1 or 2.
  Default=Undefined
  Required=No

* **blue** (Any : IN):
  Name of blue colour field in input file IN. Only used if @WFMETHOD is 1 or 2.
  Default=Undefined
  Required=No

### Parameters:

* **runmode**:
  Specify the functions for the process to execute:
  Range=
  Values=
  Default=
  Required=No

* **ssmethod**:
  The method to use for subsampling.
  Range=
  Values=
  Default=
  Required=No

* **ssdist**:
  A parameter controlling the extent to which subsampling is performed and depends on the
  chosen _@SSMETHOD_.
  Range=
  Values=
  Default=
  Required=No

* **nneighbs**:
  The number of neighbours used when orienting the normals using a spanning tree. Normals
  are not required if _@WFMETHOD_ is 3,4 or 5.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **radius**:
  Where @WFMETHOD = 1 or 2: the neighborhood radius used to compute normals. The bigger
  the radius, the more points will be used to compute the local surface model, resulting
  in generally smoother normals but also in a longer process time. A value of zero will
  allow the program to compute a radius value automatically, based on the mean average
  inter-point distance. Where @WFMETHOD = 4, this represents the radius of the sphere used
  to generate the surface by rolling across the input points. Not used if @WFMETHOD Is 3
  or 5.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **depth**:
  The octree depth to be used in surface construction. A higher number gives greater
  resolution but will be slower and use more memory. Should ideally be between 4 and 24.
  Values larger than 10 are likely to be process intensive. This parameter is ignored if
  WIDTH is specified as greater than zero. Not considered if @WFMETHOD 3,4 or 5.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **width**:
  This floating point value specifies the target width of the finest level octree cells
  used in surface reconstruction. If set to zero then DEPTH is used to control octree
  size. Not considered if @WFMETHOD 3,4 or 5.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **wfmethod**:
  The method to use to construct the wireframe surface from the set of points with
  oriented normals.
  Range=
  Values=
  Default=
  Required=No

* **sampnode**:
  Number of samples per node during Reconstruction. This floating point value specifies
  the minimum number of sample points that should fall within an octree node as the octree
  construction is adapted to sampling density. For noise-free samples, small values in the
  range [1.0 - 5.0] can be used. For more noisy samples, larger values in the range [15.0
  - 20.0] may be needed to provide a smoother, noise-reduced, reconstruction. Only used if
  @WFMETHOD is 1 or 2.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **ptweight**:
  The importance that interpolation of the point samples is given in the formulation of
  the screened Poisson equation. This is a floating point number. Only used if @WFMETHOD
  is 1 or 2.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('ptcld2wf')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute ptcld2wf
print("Running ptcld2wf...")
dm_cmd.ptcld2wf(
    in_i='t_assays',  # required
    points_o=['t_ptcld2wf_out'],  # required
    wiretr_o='t_ptcld2wf_out',  # required
    wirept_o='t_ptcld2wf_out',  # required
    # red_f='optional',  # optional
    # green_f='optional',  # optional
    # blue_f='optional',  # optional
    # runmode_p='optional',  # optional
    # ssmethod_p='optional',  # optional
    # ssdist_p='optional',  # optional
    # nneighbs_p='optional',  # optional
    # radius_p='optional',  # optional
    # depth_p='optional',  # optional
    # width_p='optional',  # optional
    # wfmethod_p='optional',  # optional
    # sampnode_p='optional',  # optional
    # ptweight_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("ptcld2wf execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_ptcld2wf_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")